# Zomato Restaurant Clustering and Customer Sentiment Analysis

##### **Project Type** - Unsupervised Machine Learning (with exploratory data analysis and sentiment analysis)

##### **Contribution** - Individual

##### **Team Member 1** - Hasini Bavirisetty

##### **Team Member 2–4** - Not applicable

# **Project Summary -**

## Project Summary

This project analyses Zomato restaurant metadata and customer reviews to understand restaurant characteristics, customer feedback and similarities between restaurants. The work combines exploratory data analysis, review sentiment analysis and unsupervised machine learning. The business objective is to turn restaurant and review data into interpretable information that may help customers compare restaurants and help businesses understand restaurant segments and customer experience.

Two datasets were used: restaurant metadata and customer reviews. The metadata includes restaurant names, links, approximate dining cost, collections and cuisines. The review dataset includes restaurant/reviewer information, review text, ratings and reviewer metadata. The datasets were cleaned and restaurant-level information was created by aggregating review ratings. Restaurant names were normalised for matching before the metadata and rating summaries were merged. The clustering analysis used three numerical features: Cost, Average_Rating and Total_Reviews. Missing or infinite values in the selected clustering features were excluded from the clustering input.

Exploratory charts examined ratings, cuisines, costs, restaurant averages, cluster patterns and review sentiment. VADER was applied to review text to classify reviews as positive, neutral or negative. In the analysed review set of 9,955 records, 7,392 (74.25%) were positive, 717 (7.20%) neutral and 1,846 (18.54%) negative. Common positive words included “good”, “food”, “place” and “service”; negative reviews included terms such as “bad”, “worst”, “experience” and “service”. These results provide a high-level view of customer opinion, but automated sentiment can miss sarcasm, context and multilingual expressions.

Three clustering algorithms were implemented: K-Means, Agglomerative Clustering and DBSCAN. The features were standardised using StandardScaler because cost, rating and review count have different scales. The final comparison used K-Means with three clusters (Silhouette Score 0.4673), Agglomerative Clustering with three clusters (0.4181), and DBSCAN (0.5098 on 56 non-noise observations). DBSCAN’s score is based only on the observations it assigned to clusters; 44 of 100 observations were labelled noise, so it is not directly comparable with the scores for the other methods. K-Means with three clusters was selected as the practical final model because it assigns all observations to three groups and provides a useful balance between separation and coverage.

Hypothesis tests examined differences in ratings across cost groups, the association between cost and average rating, and the association between numerical ratings and text-derived sentiment. The results indicated statistically significant relationships in this dataset; statistical significance does not establish causation. Limitations include the relatively small merged restaurant sample (100 restaurants), the near-constant review-count feature for many restaurants, and the limitations of rule-based sentiment analysis. Further work could add more locations, restaurants, cuisine and geographic features, test more robust text models, profile clusters in greater detail and deploy a reproducible prediction workflow.

# **GitHub Link -**

https://github.com/hasinibavirisetty15-cpu/hasinibavirisetty15-cpu-Zomato-Restaurant-Clustering

# **Problem Statement**


The project addresses the need to understand restaurant characteristics and customer opinions from Zomato data. The work combines restaurant metadata with customer reviews, explores ratings, cuisines and costs, analyses review sentiment, and groups restaurants with similar numerical characteristics using unsupervised clustering. The goal is to produce interpretable business insights that can support restaurant comparison and help businesses understand customer experience and restaurant segments. The analysis is exploratory and does not claim to predict restaurant success or prove that one factor causes another.

# ***Let's Begin !***

## 1. Know Your Data and Prepare the Datasets

The following executable sections load the two supplied CSV files, inspect them, clean their fields, and build the restaurant-level dataset used throughout the analysis. The loader checks the notebook folder and a sibling `outputs` folder, so the notebook works both from the delivered bundle and from the project root.


In [ ]:
# Step 1: Import required libraries

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

# Locate the supplied CSV files beside this notebook or in the project outputs folder.
base_dir = Path.cwd()
search_dirs = [base_dir, base_dir / "outputs", base_dir.parent / "outputs"]
def find_dataset(filename):
    for folder in search_dirs:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {filename}. Place both CSV files beside the notebook or in an outputs folder.")

reviews_path = find_dataset("Zomato Restaurant reviews.csv")
restaurants_path = find_dataset("Zomato Restaurant names and Metadata.csv")
reviews = pd.read_csv(reviews_path)
restaurants = pd.read_csv(restaurants_path)
print(f"Loaded reviews from: {reviews_path.resolve()}")
print(f"Loaded restaurant metadata from: {restaurants_path.resolve()}")
print("Both datasets loaded successfully!")

In [ ]:
print("CUSTOMER REVIEWS")
display(reviews.head())

print("RESTAURANT INFORMATION")
display(restaurants.head())

In [ ]:
# Check dataset dimensions
print("Reviews shape:", reviews.shape)
print("Restaurants shape:", restaurants.shape)

# Check column names
print("\nReviews columns:")
print(reviews.columns.tolist())

print("\nRestaurant columns:")
print(restaurants.columns.tolist())

# Check missing values
print("\nMissing values in reviews:")
print(reviews.isnull().sum())

print("\nMissing values in restaurants:")
print(restaurants.isnull().sum())

In [ ]:
print("Duplicate reviews:", reviews.duplicated().sum())
print("Duplicate restaurants:", restaurants.duplicated().sum())

## Data Preprocessing

Data preprocessing was performed to improve data quality and prepare the restaurant and customer review datasets for analysis.

The following steps were carried out:

- **Loading datasets:** Restaurant metadata and customer review datasets were imported using Pandas.
- **Removing duplicates:** Duplicate records were removed to avoid repeated data.
- **Handling missing values:** Missing review text was replaced with empty strings, and records with missing values in important clustering features were excluded.
- **Data type conversion:** Restaurant cost and customer ratings were converted into numeric formats for analysis.
- **Text cleaning:** Review text was converted to lowercase and unnecessary surrounding whitespace was removed.
- **Feature preparation:** Average restaurant rating and total review count were calculated from customer reviews and combined with restaurant cost.

In [ ]:
# Keep original datasets safe
reviews_clean = reviews.copy()
restaurants_clean = restaurants.copy()

# Clean column names
reviews_clean.columns = reviews_clean.columns.str.strip()
restaurants_clean.columns = restaurants_clean.columns.str.strip()

# Remove duplicate rows
reviews_clean = reviews_clean.drop_duplicates()
restaurants_clean = restaurants_clean.drop_duplicates()

# Remove extra spaces from text columns
for df in [reviews_clean, restaurants_clean]:
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

print("Data cleaning completed!")

print("Reviews before:", reviews.shape)
print("Reviews after:", reviews_clean.shape)

print("Restaurants before:", restaurants.shape)
print("Restaurants after:", restaurants_clean.shape)

In [ ]:
print("Missing values in reviews:")
print(reviews_clean.isnull().sum())

print("\nMissing values in restaurants:")
print(restaurants_clean.isnull().sum())

In [ ]:
# Convert ratings into numeric values
reviews_clean['Rating'] = pd.to_numeric(
    reviews_clean['Rating'], errors='coerce'
)

# Check the results
print("Rating data type:", reviews_clean['Rating'].dtype)
print("\nRating summary:")
print(reviews_clean['Rating'].describe())

print("\nMissing ratings:", reviews_clean['Rating'].isnull().sum())

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(reviews_clean['Rating'].dropna(), bins=10, kde=True)

plt.title("Distribution of Restaurant Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.show()

**Why this chart was chosen**: A histogram was used to understand how customer ratings are distributed across the restaurant reviews.

**Insights**: The chart shows that ratings of 5.0 and 4.0 are the most frequent. Ratings of 1.0 are also noticeable, indicating that some customers had poor experiences. Overall, positive ratings are more common than low ratings.

**Business impact**: Restaurants can use rating distributions to understand customer satisfaction. Businesses with lower ratings can investigate service, food quality, and customer experience issues.

In [ ]:
# Remove rows with empty review text
reviews_clean = reviews_clean.dropna(subset=['Review'])

# Convert reviews to lowercase
reviews_clean['Review'] = reviews_clean['Review'].astype(str).str.lower()

# Remove extra spaces
reviews_clean['Review'] = reviews_clean['Review'].str.strip()

print("Review text cleaning completed!")
print("Remaining reviews:", len(reviews_clean))

display(reviews_clean[['Review', 'Rating']].head(10))

In [ ]:
print(reviews_clean['Rating'].value_counts().sort_index())

In [ ]:
# Analyze popular cuisines

cuisine_counts = (
    restaurants_clean['Cuisines']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .head(10)
)

plt.figure(figsize=(10, 5))
sns.barplot(x=cuisine_counts.values, y=cuisine_counts.index)

plt.title("Top 10 Popular Cuisines")
plt.xlabel("Number of Restaurants")
plt.ylabel("Cuisine")
plt.show()

display(cuisine_counts)

**Why this chart was chosen**: A horizontal bar chart was selected to compare the number of restaurants offering different cuisines.

**Insights**: North Indian cuisine leads with 61 restaurants, followed by Chinese cuisine with 43 and Continental cuisine with 21. Biryani, Asian, and Fast Food are also popular.

**Business** **impact**: Restaurant owners can use cuisine popularity to identify market demand and decide which food categories may attract more customers.

In [ ]:
# Restaurant Cost Analysis

# Convert Cost column into numeric format
restaurants_clean['Cost'] = pd.to_numeric(
    restaurants_clean['Cost'].astype(str)
    .str.replace(',', '', regex=False)
    .str.extract(r'(\d+\.?\d*)')[0],
    errors='coerce'
)

# Display cost statistics
print("RESTAURANT COST SUMMARY")
display(restaurants_clean['Cost'].describe())

# Plot cost distribution
plt.figure(figsize=(10, 5))

sns.histplot(
    restaurants_clean['Cost'].dropna(),
    bins=15,
    kde=True
)

plt.title("Distribution of Restaurant Costs", fontsize=14)
plt.xlabel("Cost")
plt.ylabel("Number of Restaurants")
plt.tight_layout()
plt.show()

print("Cost analysis completed successfully!")

**Why this chart was chosen**: A histogram with a density curve was used to understand the distribution of approximate restaurant costs.

**Insights**: Most restaurants fall within the lower-to-middle cost range, particularly around 500–1,000. A smaller number of restaurants have costs above 1,500, and a few are above 2,500. The distribution is right-skewed.

**Business impact**: This helps identify the dominant price range in the market and supports pricing and customer-segmentation decisions.

In [ ]:
print(restaurants_clean['Cost'].describe())

In [ ]:
# Restaurant Rating Analysis

# Calculate average rating and number of reviews
restaurant_ratings = (
    reviews_clean.dropna(subset=['Rating'])
    .groupby('Restaurant')
    .agg(
        Average_Rating=('Rating', 'mean'),
        Total_Reviews=('Rating', 'count')
    )
    .reset_index()
)

# Consider restaurants with at least 3 reviews
restaurant_ratings = restaurant_ratings[
    restaurant_ratings['Total_Reviews'] >= 3
]

# Display top 10 restaurants
top_restaurants = restaurant_ratings.sort_values(
    by='Average_Rating', ascending=False
).head(10)

print("TOP 10 RESTAURANTS BY AVERAGE RATING")
display(top_restaurants)

# Visualize results
plt.figure(figsize=(11, 6))

sns.barplot(
    data=top_restaurants,
    x='Average_Rating',
    y='Restaurant'
)

plt.title("Top 10 Restaurants by Average Customer Rating")
plt.xlabel("Average Rating")
plt.ylabel("Restaurant")
plt.xlim(0, 5)
plt.tight_layout()
plt.show()

**Why this chart was chosen**: A horizontal bar chart was used to compare restaurants based on their average customer ratings.

**Insights**: AB's - Absolute Barbecues has the highest average rating at 4.88, followed by B-Dubs at 4.81 and 3B's - Buddies, Bar & Barbecue at 4.76. The selected restaurants have at least three reviews.

**Business impact**: This analysis helps identify restaurants with strong customer satisfaction that can be studied for successful practices.

In [ ]:
# Combine restaurant details and customer ratings

# Create matching restaurant names
restaurants_clean['Name_Key'] = (
    restaurants_clean['Name'].astype(str).str.strip().str.lower()
)

restaurant_ratings['Name_Key'] = (
    restaurant_ratings['Restaurant'].astype(str).str.strip().str.lower()
)

# Merge datasets
zomato = pd.merge(
    restaurants_clean,
    restaurant_ratings,
    on='Name_Key',
    how='inner'
)

# Display results
print("Combined dataset created successfully!")
print("Combined dataset shape:", zomato.shape)

display(zomato.head(10))

print("Columns:", zomato.columns.tolist())

In [ ]:
print("Restaurants in metadata:", len(restaurants_clean))
print("Restaurants with ratings:", len(restaurant_ratings))
print("Successfully matched restaurants:", len(zomato))

In [ ]:
# Check available numerical features

print("Available columns:")
print(zomato.columns.tolist())

print("\nMissing values:")
print(zomato[['Cost', 'Average_Rating', 'Total_Reviews']].isnull().sum())

print("\nNumerical feature summary:")
display(zomato[['Cost', 'Average_Rating', 'Total_Reviews']].describe())

In [ ]:
# Select features for clustering
features = ['Cost', 'Average_Rating', 'Total_Reviews']

# Create a clean dataset
cluster_data = zomato[features].copy()

# Remove missing and infinite values
cluster_data = cluster_data.replace([np.inf, -np.inf], np.nan)
cluster_data = cluster_data.dropna()

print("Clustering data prepared!")
print("Number of restaurants:", len(cluster_data))

display(cluster_data.head())

### Feature Engineering and Feature Scaling

Feature engineering was performed to create meaningful numerical features for restaurant clustering.

The following features were used:

- **Cost:** Average cost of dining at a restaurant.
- **Average Rating:** Average customer rating calculated from customer reviews.
- **Total Reviews:** Number of customer reviews available for each restaurant.

The three numerical features were selected to understand how restaurant pricing, customer satisfaction, and review activity relate to each other.

**Feature Scaling:** StandardScaler was used to standardize the numerical features so that variables with larger numerical ranges would not dominate the clustering process.

In [ ]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(cluster_data)

scaled_df = pd.DataFrame(
    scaled_features,
    columns=features,
    index=cluster_data.index
)

print("Feature scaling completed!")
display(scaled_df.head())

## 2. Understanding the Variables

The clustering uses three numeric fields: approximate restaurant cost, mean customer rating, and review count. Restaurant name is used only to join metadata to review aggregates. Review text is analyzed separately for sentiment.


## 3. Data Wrangling and Feature Preparation

Duplicate records and inconsistent text spacing are handled, ratings and costs are converted to numeric values, and review ratings are aggregated by restaurant before the datasets are joined. Rows without usable clustering features are excluded. The selected features are standardized because cost and ratings use different scales.


## 4. Exploratory Charts and Restaurant Clustering

The exploratory charts below preserve the original analysis and plots. Charts 14 and 15 are implemented here and include their business interpretation.


#### Chart 14 — Correlation Heatmap

**Why this chart:** A heatmap summarizes pairwise linear relationships among the three numeric clustering features.


In [ ]:
# Chart 14: Correlation among the clustering variables
correlation_matrix = cluster_data[features].corr(method="pearson")
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Correlation Between Restaurant Clustering Features")
plt.tight_layout()
plt.show()


**Insight and business impact:** Read the annotated values to see whether higher cost or review volume tends to move with higher ratings in this sample. A weak relationship means price or popularity alone is not a reliable proxy for satisfaction; businesses should assess these measures separately. Correlation is descriptive and does not establish cause.


## 5. Clustering Models, Evaluation, and Tuning

K-Means, Agglomerative Clustering, and DBSCAN are compared with internal clustering metrics. Since there is no labeled target, the tuning step tests candidate cluster counts using silhouette score and inertia; it does not use supervised cross-validation.


## Machine Learning Algorithms

Unsupervised machine learning techniques were used to group restaurants based on similarities in cost, average customer ratings, and review activity.

### 1. K-Means Clustering

K-Means is a centroid-based clustering algorithm that divides data points into a specified number of clusters. It assigns restaurants to groups based on their similarity to cluster centroids.

In this project, K-Means was used to identify groups of restaurants with similar pricing, customer ratings, and review activity.

### 2. Agglomerative Clustering

Agglomerative Clustering is a hierarchical clustering technique. It starts by treating each data point as an individual cluster and gradually merges similar clusters until the required number of clusters is reached.

It was used to compare hierarchical grouping results with K-Means.

### 3. DBSCAN

DBSCAN is a density-based clustering algorithm that groups closely packed data points and identifies isolated points as noise or outliers.

It was used to explore density-based restaurant groups and detect restaurants that did not fit into dense clusters.

In [ ]:
from sklearn.cluster import KMeans

inertia = []

for k in range(2, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(scaled_features)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(range(2, 11), inertia, marker='o')

plt.title("Elbow Method for Optimal Clusters")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.xticks(range(2, 11))
plt.grid(True)
plt.show()

**Why this chart was chosen**: The Elbow Method was used to examine how K-Means inertia changes as the number of clusters increases.

**Insights**: Inertia decreases as more clusters are added. The reduction becomes less pronounced after approximately three or four clusters, suggesting that adding more clusters gives smaller improvements.

**Business impact**: Selecting a reasonable number of clusters helps create useful restaurant segments without making the model unnecessarily complicated.

In [ ]:
# K-Means Clustering Model

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(scaled_features)

# Add cluster labels to the dataset
cluster_data = cluster_data.copy()
cluster_data['Cluster'] = cluster_labels

print("K-Means model trained successfully!")
print("\nRestaurants in each cluster:")
print(cluster_data['Cluster'].value_counts().sort_index())

# Evaluate model
silhouette = silhouette_score(scaled_features, cluster_labels)

print("\nSilhouette Score:", round(silhouette, 4))

#### Chart 15 — Pair Plot of K-Means Features

**Why this chart:** Pairwise scatter plots show how the three clustering features vary together, with color identifying the four-cluster exploratory K-Means assignments.


In [ ]:
# Chart 15: Pairwise feature relationships colored by the exploratory K-Means labels
pair_grid = sns.pairplot(cluster_data, vars=features, hue="Cluster", diag_kind="hist", corner=True, plot_kws={"alpha": 0.75, "s": 35})
pair_grid.fig.suptitle("Pair Plot of Restaurant Features by K-Means Cluster", y=1.02)
plt.show()


**Insight and business impact:** The pair plot reveals which feature combinations drive separation and whether groups overlap. This helps operators compare restaurants on price, rating, and review activity together, while overlap warns that cluster labels are broad profiles rather than definitive business categories.


In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=cluster_data,
    x='Cost',
    y='Average_Rating',
    hue='Cluster',
    palette='viridis',
    s=100
)

plt.title("Zomato Restaurant Clusters")
plt.xlabel("Restaurant Cost")
plt.ylabel("Average Rating")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

### Four-Cluster K-Means View

The scatter plot shows the four exploratory K-Means groups using restaurant cost and average rating. The full model also includes review count, so overlap in this two-dimensional view is expected. Cluster identifiers are arbitrary labels, not rankings. The rare group should be interpreted cautiously.


In [ ]:
# Analyze restaurant clusters

cluster_summary = cluster_data.groupby('Cluster').agg(
    Number_of_Restaurants=('Cost', 'count'),
    Average_Cost=('Cost', 'mean'),
    Average_Rating=('Average_Rating', 'mean'),
    Average_Reviews=('Total_Reviews', 'mean')
).round(2)

print("RESTAURANT CLUSTER SUMMARY")
display(cluster_summary)

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=cluster_data,
    x='Cluster',
    hue='Cluster',
    palette='viridis',
    legend=False
)

plt.title("Number of Restaurants in Each Cluster")
plt.xlabel("Cluster")
plt.ylabel("Number of Restaurants")
plt.tight_layout()
plt.show()

**Why this chart:** A count plot compares the number of restaurants assigned to each exploratory cluster.

**Insight and business impact:** The plotted counts show whether restaurant profiles are common or rare in this sample. A small group may be useful for identifying unusual profiles, but it is too small to support broad business conclusions without further validation.


## Final Cluster Analysis and Interpretation

The saved notebook text previously reported a 24 / 74 / 2 split. K-Means does not impose equal group sizes: it minimizes within-cluster distances, so the largest group captures the common feature pattern and the two-member group captures a small, distinct pattern. The two-member cluster represents only 2% of the 100 matched restaurants and should be treated as exploratory, not as a proven market segment. Inspect those restaurants individually and validate with additional data before taking action.

On a fresh run using the supplied CSVs and the notebook’s documented scaling and random seed, the tuned K=3 solution produces a 19 / 79 / 2 split. The difference from the earlier 24 / 74 / 2 text is a reproducibility issue in the original saved notebook state; the executed results in this version are authoritative. The 79-member cluster describes the dominant pattern in this sample, not customer demand or business performance.


In [ ]:
print("FINAL CLUSTER SUMMARY")

cluster_summary = cluster_data.groupby('Cluster').agg(
    Number_of_Restaurants=('Cost', 'count'),
    Average_Cost=('Cost', 'mean'),
    Average_Rating=('Average_Rating', 'mean'),
    Average_Reviews=('Total_Reviews', 'mean')
).round(2)

display(cluster_summary)

In [ ]:
# K-Means Model Evaluation

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

silhouette = silhouette_score(
    scaled_features, cluster_labels
)

davies = davies_bouldin_score(
    scaled_features, cluster_labels
)

calinski = calinski_harabasz_score(
    scaled_features, cluster_labels
)

print("K-MEANS EVALUATION RESULTS")
print("--------------------------------")
print("Silhouette Score:", round(silhouette, 4))
print("Davies-Bouldin Score:", round(davies, 4))
print("Calinski-Harabasz Score:", round(calinski, 4))

In [ ]:
# Hierarchical Clustering

from sklearn.cluster import AgglomerativeClustering

hierarchical = AgglomerativeClustering(
    n_clusters=4,
    linkage='ward'
)

hierarchical_labels = hierarchical.fit_predict(scaled_features)

# Evaluate Hierarchical Clustering
h_silhouette = silhouette_score(
    scaled_features, hierarchical_labels
)

h_davies = davies_bouldin_score(
    scaled_features, hierarchical_labels
)

h_calinski = calinski_harabasz_score(
    scaled_features, hierarchical_labels
)

print("HIERARCHICAL CLUSTERING RESULTS")
print("--------------------------------")
print("Silhouette Score:", round(h_silhouette, 4))
print("Davies-Bouldin Score:", round(h_davies, 4))
print("Calinski-Harabasz Score:", round(h_calinski, 4))

In [ ]:
# DBSCAN Clustering

from sklearn.cluster import DBSCAN

eps_values = [0.3, 0.5, 0.7, 0.9, 1.1, 1.3]

print("DBSCAN PARAMETER ANALYSIS")
print("-" * 50)

for eps in eps_values:

    dbscan = DBSCAN(eps=eps, min_samples=5)

    labels = dbscan.fit_predict(scaled_features)

    clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise = list(labels).count(-1)

    print(f"\nEpsilon: {eps}")
    print("Number of clusters:", clusters)
    print("Noise points:", noise)

    # Calculate silhouette only when valid
    valid_labels = labels[labels != -1]
    valid_data = scaled_features[labels != -1]

    if len(set(valid_labels)) >= 2 and len(valid_labels) > len(set(valid_labels)):
        score = silhouette_score(valid_data, valid_labels)
        print("Silhouette Score (excluding noise):", round(score, 4))
    else:
        print("Silhouette Score: Not applicable")

In [ ]:
# Final DBSCAN Model

dbscan_final = DBSCAN(eps=0.3, min_samples=5)

dbscan_labels = dbscan_final.fit_predict(scaled_features)

# Add DBSCAN labels
dbscan_results = cluster_data.copy()
dbscan_results['DBSCAN_Cluster'] = dbscan_labels

print("DBSCAN CLUSTER DISTRIBUTION")
print(dbscan_results['DBSCAN_Cluster'].value_counts().sort_index())

# Visualize clusters and noise
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=dbscan_results,
    x='Cost',
    y='Average_Rating',
    hue='DBSCAN_Cluster',
    palette='tab10',
    s=100
)

plt.title("DBSCAN Restaurant Clustering (Epsilon = 0.3)")
plt.xlabel("Restaurant Cost")
plt.ylabel("Average Rating")
plt.legend(title="Cluster / Noise")
plt.tight_layout()
plt.show()

**Why this chart was chosen**: A scatter plot was used to visualise the clusters identified by DBSCAN and distinguish noise points from clustered restaurants.

**Insights**: With epsilon set to 0.3 and minimum samples set to 5, DBSCAN identifies five clusters and 44 noise points. The noise points indicate restaurants that do not fit closely into the identified dense groups.

**Business impact**: DBSCAN helps identify unusual restaurant profiles that may require separate investigation.

## Model Evaluation and Comparison

The clustering algorithms were evaluated using internal clustering evaluation metrics to understand the quality and separation of the generated clusters.

The following metrics were considered:

- **Silhouette Score:** Measures how similar a data point is to its own cluster compared with other clusters. A higher score indicates better-defined clusters.
- **Davies-Bouldin Index:** Measures the average similarity between clusters. A lower score indicates better cluster separation.
- **Calinski-Harabasz Index:** Measures the ratio of between-cluster dispersion to within-cluster dispersion. A higher score generally indicates better-defined clusters.

For a fair comparison, K-Means and Agglomerative Clustering are evaluated with K=3 using all restaurants. DBSCAN metrics are calculated only for non-noise observations because DBSCAN labels outliers as noise (-1).

**Note:** The DBSCAN silhouette score was calculated only for non-noise observations, so it should not be directly compared with scores calculated using all observations for the other algorithms.

In [ ]:
# Fair model comparison: K-Means K=3, Agglomerative K=3, and DBSCAN
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# Fit both partition-based models using the same scaled observations and K=3.
kmeans_k3_labels = KMeans(
    n_clusters=3, random_state=42, n_init=10
).fit_predict(scaled_features)

agglo_k3_labels = AgglomerativeClustering(
    n_clusters=3, linkage='ward'
).fit_predict(scaled_features)

def calculate_metrics(data, labels):
    # Internal clustering metrics require at least 2 clusters and fewer
    # clusters than observations.
    if len(set(labels)) < 2 or len(set(labels)) >= len(labels):
        return (np.nan, np.nan, np.nan)
    return (
        silhouette_score(data, labels),
        davies_bouldin_score(data, labels),
        calinski_harabasz_score(data, labels)
    )

km_sil, km_db, km_ch = calculate_metrics(scaled_features, kmeans_k3_labels)
ag_sil, ag_db, ag_ch = calculate_metrics(scaled_features, agglo_k3_labels)

# DBSCAN metrics are calculated only on observations that are not labelled noise.
dbscan_mask = dbscan_labels != -1
dbscan_non_noise_labels = dbscan_labels[dbscan_mask]
db_sil, db_db, db_ch = calculate_metrics(
    scaled_features[dbscan_mask], dbscan_non_noise_labels
)

comparison = pd.DataFrame([
    {
        'Model': 'K-Means (K=3)',
        'Silhouette Score': km_sil,
        'Davies-Bouldin Score': km_db,
        'Calinski-Harabasz Score': km_ch,
        'Evaluation Set': 'All restaurants'
    },
    {
        'Model': 'Agglomerative (K=3)',
        'Silhouette Score': ag_sil,
        'Davies-Bouldin Score': ag_db,
        'Calinski-Harabasz Score': ag_ch,
        'Evaluation Set': 'All restaurants'
    },
    {
        'Model': 'DBSCAN (eps=0.3)',
        'Silhouette Score': db_sil,
        'Davies-Bouldin Score': db_db,
        'Calinski-Harabasz Score': db_ch,
        'Evaluation Set': f'Non-noise restaurants only ({dbscan_mask.sum()} of {len(dbscan_mask)})'
    }
])

print("CLUSTERING MODEL COMPARISON")
display(comparison.round(4))
print(
    "Note: DBSCAN metrics exclude noise observations (-1), so they are not "
    "directly comparable with K-Means and Agglomerative metrics calculated "
    "using all restaurants."
)


## 6. Customer Review Sentiment Analysis

VADER labels the review text as positive, neutral, or negative. Its lexicon-based results are useful for an overview, but can miss sarcasm, context, and local-language expressions.


In [ ]:
# Customer Review Sentiment Analysis

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    score = analyzer.polarity_scores(str(text))['compound']

    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

reviews_clean['Sentiment'] = reviews_clean['Review'].apply(get_sentiment)

print("SENTIMENT ANALYSIS COMPLETED")
print(reviews_clean['Sentiment'].value_counts())

display(reviews_clean[['Review', 'Sentiment']].head(10))

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=reviews_clean,
    x='Sentiment',
    hue='Sentiment',
    order=['Positive', 'Neutral', 'Negative'],
    palette='viridis',
    legend=False
)

plt.title("Customer Review Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

**Why this chart was chosen**: A count plot was used to compare positive, negative, and neutral customer reviews identified using VADER sentiment analysis.

**Insights**: Out of 9,955 reviews, 7,392 (74.25%) are positive, 1,846 (18.54%) are negative, and 717 (7.20%) are neutral. Positive reviews form the largest category.

**Business impact**: Restaurants can monitor customer sentiment to understand overall satisfaction and identify areas requiring improvement.

In [ ]:
sentiment_counts = reviews_clean['Sentiment'].value_counts()

sentiment_percentages = (
    reviews_clean['Sentiment'].value_counts(normalize=True) * 100
)

sentiment_summary = pd.DataFrame({
    'Number of Reviews': sentiment_counts,
    'Percentage': sentiment_percentages.round(2)
})

display(sentiment_summary)

In [ ]:
restaurant_sentiment = pd.crosstab(
    reviews_clean['Restaurant'],
    reviews_clean['Sentiment']
)

restaurant_sentiment['Total_Reviews'] = restaurant_sentiment.sum(axis=1)

display(restaurant_sentiment.head(10))

In [ ]:
# Calculate sentiment percentages for each restaurant
restaurant_sentiment['Positive_Percentage'] = (
    restaurant_sentiment['Positive'] /
    restaurant_sentiment['Total_Reviews'] * 100
)

restaurant_sentiment['Negative_Percentage'] = (
    restaurant_sentiment['Negative'] /
    restaurant_sentiment['Total_Reviews'] * 100
)

# Consider restaurants with at least 50 reviews
filtered_restaurants = restaurant_sentiment[
    restaurant_sentiment['Total_Reviews'] >= 50
]

# Top 10 restaurants by positive sentiment
top_positive = filtered_restaurants.sort_values(
    'Positive_Percentage', ascending=False
).head(10)

# Top 10 restaurants by negative sentiment
top_negative = filtered_restaurants.sort_values(
    'Negative_Percentage', ascending=False
).head(10)

print("TOP 10 POSITIVE RESTAURANTS")
display(top_positive)

print("TOP 10 NEGATIVE RESTAURANTS")
display(top_negative)

In [ ]:
import matplotlib.pyplot as plt

# Create two charts
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top positive restaurants
top_positive_sorted = top_positive.sort_values(
    'Positive_Percentage'
)

axes[0].barh(
    top_positive_sorted.index,
    top_positive_sorted['Positive_Percentage']
)

axes[0].set_title('Top 10 Restaurants - Positive Sentiment')
axes[0].set_xlabel('Positive Reviews (%)')
axes[0].set_xlim(0, 100)

# Top negative restaurants
top_negative_sorted = top_negative.sort_values(
    'Negative_Percentage'
)

axes[1].barh(
    top_negative_sorted.index,
    top_negative_sorted['Negative_Percentage']
)

axes[1].set_title('Top 10 Restaurants - Negative Sentiment')
axes[1].set_xlabel('Negative Reviews (%)')
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

**Why this chart was chosen**: Two horizontal bar charts were used to compare restaurants with the highest positive-review percentages and the highest negative-review percentages.

**Insight**: AB's - Absolute Barbecues and The Indi Grill are among the restaurants with very high positive sentiment. Hotel Zara Hi-Fi has the highest negative-review percentage among the displayed restaurants, followed by Asian Meal Box and Pakwaan Grand.

**Business impact**: Restaurants with positive sentiment can be studied for successful customer-experience practices. Restaurants with higher negative sentiment can investigate customer complaints and improve service quality.

In [ ]:
rating_sentiment = pd.crosstab(
    reviews_clean['Rating'],
    reviews_clean['Sentiment']
)

display(rating_sentiment)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create rating and sentiment table
rating_sentiment = pd.crosstab(
    reviews_clean['Rating'],
    reviews_clean['Sentiment']
)

# Plot heatmap
plt.figure(figsize=(10, 7))

sns.heatmap(
    rating_sentiment,
    annot=True,
    fmt='d',
    cmap='YlGnBu',
    linewidths=0.5
)

plt.title('Customer Rating vs Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Customer Rating')

plt.tight_layout()
plt.show()

**Why this chart was chosen**: A heatmap was selected to show the relationship between numerical customer ratings and text-based sentiment categories.

**Insights**: Higher ratings are strongly associated with positive sentiment. For example, rating 5.0 has 3,600 positive reviews, while rating 1.0 has 1,189 negative reviews. This indicates a clear relationship between customer ratings and review sentiment.

**Business impact**: Combining numerical ratings with written feedback provides a better understanding of customer satisfaction.

In [ ]:
# Convert counts into percentages within each rating
rating_percentage = rating_sentiment.div(
    rating_sentiment.sum(axis=1),
    axis=0
) * 100

plt.figure(figsize=(10, 7))

sns.heatmap(
    rating_percentage,
    annot=True,
    fmt='.1f',
    cmap='YlGnBu',
    linewidths=0.5
)

plt.title('Customer Rating vs Sentiment Percentage')
plt.xlabel('Sentiment')
plt.ylabel('Customer Rating')

plt.tight_layout()
plt.show()


**Why this chart was chosen**: A percentage heatmap was used to compare sentiment proportions within each customer-rating group.

**Insights**: For ratings of 4.0, 94.1% of reviews are positive. For ratings of 5.0, 94.1% are positive. In comparison, 68.5% of rating-1.0 reviews are negative.

**Business impact**: Percentage comparisons help restaurants understand the sentiment pattern within each rating group rather than relying only on the number of reviews.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt

# Separate positive and negative reviews
positive_reviews = reviews_clean[
    reviews_clean['Sentiment'] == 'Positive'
]['Review'].dropna()

negative_reviews = reviews_clean[
    reviews_clean['Sentiment'] == 'Negative'
]['Review'].dropna()

# Function to find frequently used words
def get_common_words(reviews):
    vectorizer = CountVectorizer(
        stop_words='english',
        max_features=15,
        token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'
    )

    word_matrix = vectorizer.fit_transform(reviews)

    word_counts = word_matrix.sum(axis=0).A1

    words = vectorizer.get_feature_names_out()

    return pd.Series(word_counts, index=words).sort_values()

# Calculate common words
positive_words = get_common_words(positive_reviews)
negative_words = get_common_words(negative_reviews)

# Plot positive and negative words
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

positive_words.plot(kind='barh', ax=axes[0])
axes[0].set_title('Most Common Words in Positive Reviews')
axes[0].set_xlabel('Frequency')

negative_words.plot(kind='barh', ax=axes[1])
axes[1].set_title('Most Common Words in Negative Reviews')
axes[1].set_xlabel('Frequency')

plt.tight_layout()
plt.show()

**Why this chart was chosen**: Two horizontal bar charts were used to identify frequently occurring words in positive and negative customer reviews.

**Insights**: Positive reviews frequently contain words such as "good", "food", "place", and "service". Negative reviews also contain words such as "bad", "worst", "experience", and "service". Some words appear in both groups, so they must be interpreted in context.

**Business impact**: Frequent words help restaurants identify what customers commonly discuss and which aspects of the dining experience deserve attention.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

negative_reviews = reviews_clean[
    reviews_clean['Sentiment'] == 'Negative'
]['Review'].dropna()

vectorizer = CountVectorizer(
    stop_words='english',
    ngram_range=(2, 2),
    max_features=15
)

X = vectorizer.fit_transform(negative_reviews)

phrase_counts = X.sum(axis=0).A1
phrases = vectorizer.get_feature_names_out()

top_phrases = pd.Series(
    phrase_counts,
    index=phrases
).sort_values(ascending=False)

display(top_phrases.to_frame('Frequency'))

In [ ]:
# Search negative reviews containing complaint-related phrases

complaint_phrases = [
    'worst experience',
    'bad experience',
    'waste money',
    'worst food',
    'worst service',
    'food quality'
]

for phrase in complaint_phrases:
    print("\nCOMPLAINT:", phrase.upper())

    matching_reviews = reviews_clean[
        (reviews_clean['Sentiment'] == 'Negative') &
        (reviews_clean['Review'].str.contains(
            phrase, case=False, na=False
        ))
    ]

    display(matching_reviews[['Restaurant', 'Review']].head(3))

In [ ]:
import re

negative_reviews = reviews_clean[
    reviews_clean['Sentiment'] == 'Negative'
].copy()

complaint_categories = {
    'Food Quality': r'food quality|worst food|bad food|taste|stale',
    'Service': r'worst service|bad service|slow service|poor service',
    'Delivery': r'delivery|late delivery|delayed|cold food',
    'Price and Value': r'waste money|overpriced|expensive|price',
    'Food Quantity': r'quantity|less quantity|small portion'
}

category_counts = {}

for category, keywords in complaint_categories.items():
    matches = negative_reviews['Review'].str.contains(
        keywords, case=False, na=False, regex=True
    )
    category_counts[category] = matches.sum()

complaint_summary = pd.DataFrame(
    list(category_counts.items()),
    columns=['Complaint Category', 'Review Count']
).sort_values('Review Count', ascending=False)

display(complaint_summary)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.barplot(
    data=complaint_summary,
    x='Review Count',
    y='Complaint Category'
)

plt.title('Customer Complaint Category Distribution')
plt.xlabel('Number of Keyword Matches')
plt.ylabel('Complaint Category')

plt.tight_layout()
plt.show()

**Why this chart was chosen**: A horizontal bar chart was used to compare the frequency of keyword matches across different complaint categories.

**Insights**: Food Quality has the highest number of keyword matches at 536, followed by Price and Value at 135, Delivery at 104, Service at 102, and Food Quantity at 98.

**Business impact**: Food quality appears to be the most prominent complaint area in this keyword-based analysis. Restaurants can prioritise food consistency and quality improvements.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(
    data=restaurants_clean,
    x='Cost',
    bins=30,
    kde=True
)

plt.title('Distribution of Restaurant Costs')
plt.xlabel('Approximate Restaurant Cost')
plt.ylabel('Number of Restaurants')

plt.tight_layout()
plt.show()

**Additional Cost Distribution View** : This histogram uses more bins to provide a more detailed view of restaurant pricing. It confirms that most restaurants fall within the lower-to-middle cost ranges, while fewer restaurants belong to the high-cost range. This additional view helps examine the price distribution in greater detail.

In [ ]:
# Split cuisines and count individual cuisine types
cuisine_data = (
    restaurants_clean['Cuisines']
    .dropna()
    .str.split(',')
    .explode()
    .str.strip()
)

top_cuisines = cuisine_data.value_counts().head(15)

# Plot
plt.figure(figsize=(11, 7))

sns.barplot(
    x=top_cuisines.values,
    y=top_cuisines.index
)

plt.title('Top 15 Most Common Cuisines')
plt.xlabel('Number of Restaurants')
plt.ylabel('Cuisine')

plt.tight_layout()
plt.show()

**Why this chart was chosen**: A horizontal bar chart was used to expand the cuisine comparison from the top 10 to the top 15 categories.

**Insights**: North Indian and Chinese cuisines remain the most common categories, followed by Continental and Biryani. Other categories include Asian, Fast Food, Italian, Desserts, South Indian, Bakery, Mughlai, Kebab, Andhra, Cafe, and American.

**Business impact**: A wider cuisine comparison provides more information about restaurant variety and helps identify additional market opportunities.

In [ ]:
plt.figure(figsize=(10, 6))

sns.countplot(
    data=reviews_clean,
    x='Rating',
    order=sorted(reviews_clean['Rating'].dropna().unique())
)

plt.title('Distribution of Customer Ratings')
plt.xlabel('Customer Rating')
plt.ylabel('Number of Reviews')

plt.tight_layout()
plt.show()

**Why this chart was chosen**: A count plot was used to display the frequency of each customer-rating value.

**Insights**: Ratings of 5.0 and 4.0 occur most frequently, while ratings of 1.0 also form a noticeable group. Half-star ratings are much less frequent.

**Business impact**: This helps businesses monitor customer satisfaction and understand the distribution of feedback.

In [ ]:
plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=cluster_data,
    x='Cost',
    y='Average_Rating',
    hue='Cluster',
    palette='Set2',
    s=100
)

plt.title('K-Means Restaurant Clustering')
plt.xlabel('Restaurant Cost')
plt.ylabel('Average Customer Rating')
plt.legend(title='Cluster')

plt.tight_layout()
plt.show()

**Why this chart was chosen**: A second scatter plot was used to display the K-Means restaurant clusters in another section of the notebook.

**Insights**: The chart shows restaurant groups with different cost and average-rating combinations. It provides another view of the separation between lower-cost, moderate-rating restaurants and higher-cost, higher-rating restaurants.

**Business impact**: This visualisation supports the use of restaurant segments for competitor comparison and pricing strategies.

**Hypothesis 3: Relationship Between Customer Rating and Review Sentiment**

**Null Hypothesis (H₀):** Customer ratings and text-based review sentiment are independent of each other.

**Alternative Hypothesis (H₁):** Customer ratings and text-based review sentiment are associated with each other.

**Test selected:** Chi-Square Test of Independence.

**Reason for selection:** This test is suitable for examining whether two categorical variables are significantly associated.

**Result:** Chi-square statistic = 5134.8071, degrees of freedom = 16, p-value < 0.001.

**Conclusion:** Since the p-value is below 0.05, the null hypothesis is rejected. There is a statistically significant association between customer ratings and text-based review sentiment. However, this does not establish causation.

## 7. Hypothesis Testing

These tests examine associations and group differences in the observed data. Statistical significance does not imply causation.


**Hypothesis Testing**

Hypothesis testing is used to determine whether relationships and differences observed in the restaurant dataset are statistically significant. Three statistical tests were performed to investigate restaurant pricing, customer ratings, and review sentiment.

### Hypothesis 1: Restaurant Cost and Average Rating

**Null Hypothesis (H₀):** There is no statistically significant difference in average restaurant ratings across different restaurant cost groups.

**Alternative Hypothesis (H₁):** At least one restaurant cost group has a significantly different average rating.

**Test selected:** Kruskal-Wallis H-test.

**Reason for selection:** This non-parametric test compares three or more independent groups and does not require the data to follow a normal distribution.

**Result:** H-statistic = 17.6445, p-value = 0.0001474.

**Conclusion:** Since the p-value is below 0.05, the null hypothesis is rejected. There is a statistically significant difference in ratings across the restaurant cost groups.

In [ ]:
import pandas as pd
from scipy.stats import kruskal

# Prepare clean data
test_data = cluster_data[['Cost', 'Average_Rating']].dropna().copy()

# Divide restaurants into three price groups
test_data['Price_Group'] = pd.qcut(
    test_data['Cost'].rank(method='first'),
    q=3,
    labels=['Low Cost', 'Medium Cost', 'High Cost']
)

# Display group sizes and median ratings
print("PRICE GROUP SUMMARY")
display(
    test_data.groupby('Price_Group', observed=True)['Average_Rating']
    .agg(['count', 'median', 'mean'])
)

# Perform Kruskal-Wallis test
groups = [
    test_data.loc[
        test_data['Price_Group'] == group,
        'Average_Rating'
    ]
    for group in ['Low Cost', 'Medium Cost', 'High Cost']
]

statistic, p_value = kruskal(*groups)

print("\nKRUSKAL-WALLIS TEST: COST VS RATING")
print("Test statistic:", round(statistic, 4))
print("P-value:", p_value)

if p_value < 0.05:
    print("Conclusion: There is a statistically significant difference.")
else:
    print("Conclusion: No statistically significant difference was found.")

**Hypothesis 2: Relationship Between Restaurant Cost and Average Rating**

**Null Hypothesis (H₀):** There is no monotonic relationship between restaurant cost and average customer rating.

**Alternative Hypothesis (H₁):** There is a monotonic relationship between restaurant cost and average customer rating.

**Test selected:** Spearman's rank correlation.

**Reason for selection:** Spearman's correlation measures the strength and direction of a monotonic relationship and does not require the variables to be normally distributed.

**Result:** Spearman correlation = 0.3826, p-value = 0.0000854.

**Conclusion:** Since p < 0.05, reject the null hypothesis. There is a statistically significant positive monotonic association between restaurant cost and average rating. This indicates that higher-cost restaurants tend to have higher ratings in this dataset, but it does not establish causation.

In [ ]:
from scipy.stats import spearmanr

correlation, p_value = spearmanr(
    cluster_data['Cost'],
    cluster_data['Average_Rating']
)

print("Spearman correlation:", round(correlation, 4))
print("P-value:", p_value)

if p_value < 0.05:
    print("Conclusion: Significant relationship between cost and rating.")
else:
    print("Conclusion: No significant relationship between cost and rating.")

In [ ]:
from scipy.stats import chi2_contingency

# Create contingency table
rating_sentiment = pd.crosstab(
    reviews_clean['Rating'],
    reviews_clean['Sentiment']
)

# Perform Chi-square test
chi2, p_value, dof, expected = chi2_contingency(
    rating_sentiment
)

print("CHI-SQUARE TEST: RATING VS SENTIMENT")
print("-------------------------------------")
print("Chi-square statistic:", round(chi2, 4))
print("P-value:", p_value)
print("Degrees of freedom:", dof)

if p_value < 0.05:
    print("Conclusion: There is a statistically significant association.")
else:
    print("Conclusion: No statistically significant association.")

## Hyperparameter Search and Final Model Selection

Candidate K values from 2 through 8 are evaluated on the same scaled features using silhouette score and inertia. The selected value is computed from the results table below, so the notebook does not rely on a hard-coded score or claim cross-validation.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_data)

# Test different numbers of clusters
results = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(scaled_features)

    score = silhouette_score(scaled_features, labels)

    results.append({
        'Number of Clusters': k,
        'Silhouette Score': round(score, 4),
        'Inertia': round(model.inertia_, 2)
    })

tuning_results = pd.DataFrame(results)
print(tuning_results)

best_k = tuning_results.loc[
    tuning_results['Silhouette Score'].idxmax(),
    'Number of Clusters'
]

print("\nBest number of clusters:", best_k)

In [ ]:
from sklearn.cluster import KMeans

kmeans_final = KMeans(
    n_clusters=int(best_k),
    random_state=42,
    n_init=10
)

cluster_data['Cluster'] = kmeans_final.fit_predict(scaled_features)

print("Final K-Means Silhouette Score:",
      round(silhouette_score(scaled_features, cluster_data['Cluster']), 4))

print("\nCluster Sizes:")
print(cluster_data['Cluster'].value_counts().sort_index())

print("\nCluster Profiles:")
print(cluster_data.groupby('Cluster').mean().round(2))

In [ ]:
# Display the model comparison calculated above using consistent model settings.
print("FINAL CLUSTERING MODEL COMPARISON")
display(comparison.round(4))


### Clustering Model Comparison and Interpretation

K-Means and Agglomerative Clustering are compared on all restaurants using three clusters. DBSCAN metrics use non-noise observations only, so they are not directly comparable. The tuned K-Means model is selected for interpretable segments; review the measured scores above before drawing conclusions.


# Final Conclusion

## Project: Zomato Restaurant Clustering and Sentiment Analysis

This project used restaurant metadata and customer reviews to analyse restaurant performance, identify customer preferences, and discover meaningful restaurant groups.

Data cleaning, preprocessing, feature engineering, exploratory data analysis, and sentiment analysis were performed to prepare the data for machine learning.

Three unsupervised clustering algorithms were explored: K-Means, Agglomerative Clustering, and DBSCAN. After evaluating and tuning the models, K-Means with three clusters was selected, with its measured silhouette score reported in the executed results above.

The clusters revealed differences in restaurant cost, average customer ratings, and review counts. Sentiment analysis results are reported in the executed sentiment summary above.

Three statistical tests were used to investigate relationships between restaurant ratings, costs, and customer review sentiment. The results provided evidence of significant relationships in the analysed dataset.

## Business Insights

1. Restaurants can use customer sentiment to understand customer satisfaction and identify areas needing improvement.
2. Restaurant clusters can help businesses understand differences in pricing and customer ratings.
3. Negative review patterns can highlight common complaints related to food quality, pricing, delivery, and service.
4. Businesses can use these insights to improve customer experience and make data-driven decisions.

## Limitations

- The analysis was based on the available dataset and may not represent every Zomato restaurant.
- The clustering dataset contained only 100 merged restaurants.
- The Total_Reviews feature showed very little variation across restaurants.
- DBSCAN classified several observations as noise.
- VADER sentiment analysis may not fully understand sarcasm, mixed-language reviews, or local expressions.
- The cluster containing only two restaurants should be interpreted cautiously.

## Future Scope

Future work could include a larger restaurant dataset, additional features such as location and cuisine, improved multilingual sentiment analysis, and more advanced clustering techniques.

## Overall Conclusion

The project demonstrates how unsupervised machine learning and sentiment analysis can help identify restaurant segments, understand customer opinions, and generate useful business insights from restaurant data.